# Namma Space — Autonomous 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Namma Space Challenge (Round 1)**

This notebook trains an authentic 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### Pre-Flight Checklist:
1. **Runtime Type**: In the top menu, go to **Runtime → Change runtime type** → select **T4 GPU** → click **Save**.
2. **Uploaded Video**: A 45–75 second MP4/MOV recorded following `docs/CAPTURE_SOP.md` (1080p landscape, locked AE/AF, dual-loop).

### Step 1: Initialize Python 3.10 Environment (Bypasses Python 3.13 CUDA / C++ Compilation Errors)
Modern Colab defaults to Python 3.13, which breaks legacy CUDA extensions. This cell automatically sets up Python 3.10.

In [ ]:
# Install CondaColab to switch to Python 3.10
!pip install -q condacolab
import condacolab
condacolab.install()

# NOTE: Colab will display 'Your session crashed and has recovered'.
# This is completely normal and means the Python 3.10 environment has initialized!

### Step 2: Install PyTorch, Pre-Compiled CUDA Rasterizers & Roomsplat (Instant — No C++ Compilation!)
This cell installs official pre-compiled binary wheels for `diff-gaussian-rasterization` and `simple-knn` directly from HuggingFace, completely eliminating nvcc compiler errors.

In [ ]:
import os, sys

# 1. Install PyTorch with CUDA in Python 3.10
!conda install -y pytorch torchvision pytorch-cuda=11.8 -c pytorch -c nvidia

# 2. Install pre-compiled binary wheels for diff-gaussian-rasterization and simple-knn (instant!)
!pip install https://huggingface.co/spaces/dylanebert/LGM-mini/resolve/main/wheel/diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl
!pip install https://huggingface.co/spaces/dylanebert/LGM-mini/resolve/main/wheel/simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl

# 3. Clone and install Roomsplat
%cd /content
!git clone https://github.com/Ahmedn1/roomsplat.git
%cd /content/roomsplat
!pip install -e . pycolmap typer rich opencv-python-headless plyfile tqdm scipy

# 4. Clone official Inria gaussian-splatting and link config
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!python -c "from roomsplat.cli import app; app()" config --gs-dir /content/gaussian-splatting

print("\nSUCCESS: Python 3.10 environment ready with pre-compiled CUDA rasterizers!")

### Step 3: Run 3D Gaussian Splatting Reconstruction (Frame Extraction → COLMAP Poses → 30,000 Iterations)

In [ ]:
%cd /content
import os, glob
from pathlib import Path
from roomsplat.cli import run

# Auto-detect uploaded video
vids = [f for f in glob.glob("/content/*") if f.lower().endswith((".mp4", ".mov", ".m4v")) and not f.startswith("/content/sample_data")]
if not vids:
    from google.colab import files
    print("No video detected in /content! Please select your room walkthrough video to upload:")
    uploaded = files.upload()
    vids = list(uploaded.keys())
    if not vids:
        raise FileNotFoundError("Please upload a video file to proceed.")

video_path = Path(vids[0])
print(f"Target Video: {video_path} ({os.path.getsize(video_path)/(1024*1024):.2f} MB)")

print("\n=======================================================")
print("  STARTING 3D GAUSSIAN SPLATTING RECONSTRUCTION")
print("  30,000 Iterations | 1600px | Low-VRAM Mode on T4 GPU")
print("=======================================================\n")

run(
    video=video_path,
    name="Namma Space Authentic Room",
    fps=1.5,
    max_dim=1600,
    iterations=30000,
    low_vram=True,
    no_view=True
)

print("\nSUCCESS: 3D Gaussian Splatting reconstruction finished!")

### Step 4: Download `scene.splat` and `cameras.json` for Namma Space Web Viewer

In [ ]:
import glob, shutil, os
from google.colab import files

splat_dirs = glob.glob("/content/*_splat")
if splat_dirs:
    out_dir = splat_dirs[0]
    print(f"Found reconstruction output at: {out_dir}")
    
    splat_file = f"{out_dir}/scene.splat"
    cameras_file = f"{out_dir}/cameras.json"
    
    if os.path.exists(splat_file) and os.path.exists(cameras_file):
        shutil.copy(splat_file, "/content/scene.splat")
        shutil.copy(cameras_file, "/content/cameras.json")
        
        # Zip together for easy download
        !zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json
        
        print("\nDownloading namma_space_assets.zip to your computer...")
        files.download("/content/namma_space_assets.zip")
    else:
        print(f"Error: scene.splat or cameras.json not found in {out_dir}!")
else:
    print("Error: Output splat directory not found! Did training complete successfully?")